# Hidden Markov Models from Scratch

## Learning Objectives
By completing this notebook, you will:
1. Understand HMM components: states, transitions, emissions
2. Implement forward-backward algorithm for inference
3. Code Viterbi algorithm for most likely state sequence
4. Apply Baum-Welch algorithm for parameter learning
5. Detect commodity market regimes with HMMs

## Prerequisites
- Probability theory and conditional distributions
- Python programming with NumPy
- Time series concepts

## Estimated Time: 70 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 6)

print('Environment ready for HMM implementation')

## 1. HMM Theory

### Components

1. **Hidden States** $S_t \in \{1, 2, ..., K\}$: Unobserved market regime
2. **Observations** $O_t$: Observed commodity prices/returns
3. **Transition Matrix** $A$: $A_{ij} = P(S_{t+1}=j | S_t=i)$
4. **Emission Probabilities** $B$: $P(O_t | S_t=k)$
5. **Initial Distribution** $\pi$: $P(S_1=k)$

### Three Fundamental Problems

1. **Evaluation**: $P(O | \lambda)$ - Forward algorithm
2. **Decoding**: $\arg\max_S P(S | O, \lambda)$ - Viterbi algorithm  
3. **Learning**: $\arg\max_\lambda P(O | \lambda)$ - Baum-Welch algorithm

In [ ]:
class HiddenMarkovModel:
    """HMM implementation from scratch."""
    
    def __init__(self, n_states, n_obs=None):
        self.n_states = n_states
        self.n_obs = n_obs
        self.transition_matrix = None
        self.emission_params = None
        self.initial_probs = None
    
    def initialize_random(self):
        """Random initialization."""
        # Transition matrix (row-stochastic)
        self.transition_matrix = np.random.dirichlet([1]*self.n_states, self.n_states)
        
        # Initial distribution
        self.initial_probs = np.random.dirichlet([1]*self.n_states)
        
        # Emission parameters (Gaussian: mean, std per state)
        self.emission_params = {
            'means': np.random.randn(self.n_states),
            'stds': np.random.gamma(2, 0.5, self.n_states)
        }
        
        return self
    
    def emission_prob(self, obs, state):
        """P(obs | state) for Gaussian emissions."""
        mu = self.emission_params['means'][state]
        sigma = self.emission_params['stds'][state]
        return stats.norm.pdf(obs, mu, sigma)
    
    def forward(self, observations):
        """Forward algorithm: compute P(O_1:t, S_t=k)."""
        T = len(observations)
        alpha = np.zeros((T, self.n_states))
        
        # Initialization
        for k in range(self.n_states):
            alpha[0, k] = self.initial_probs[k] * self.emission_prob(observations[0], k)
        
        # Recursion
        for t in range(1, T):
            for k in range(self.n_states):
                alpha[t, k] = self.emission_prob(observations[t], k) * \
                              np.sum(alpha[t-1] * self.transition_matrix[:, k])
        
        return alpha
    
    def backward(self, observations):
        """Backward algorithm: compute P(O_{t+1:T} | S_t=k)."""
        T = len(observations)
        beta = np.zeros((T, self.n_states))
        
        # Initialization
        beta[T-1] = 1.0
        
        # Recursion (backward)
        for t in range(T-2, -1, -1):
            for k in range(self.n_states):
                for j in range(self.n_states):
                    beta[t, k] += self.transition_matrix[k, j] * \
                                   self.emission_prob(observations[t+1], j) * \
                                   beta[t+1, j]
        
        return beta
    
    def viterbi(self, observations):
        """Viterbi algorithm: find most likely state sequence."""
        T = len(observations)
        delta = np.zeros((T, self.n_states))
        psi = np.zeros((T, self.n_states), dtype=int)
        
        # Initialization
        for k in range(self.n_states):
            delta[0, k] = np.log(self.initial_probs[k]) + \
                          np.log(self.emission_prob(observations[0], k) + 1e-10)
        
        # Recursion
        for t in range(1, T):
            for k in range(self.n_states):
                probs = delta[t-1] + np.log(self.transition_matrix[:, k] + 1e-10)
                psi[t, k] = np.argmax(probs)
                delta[t, k] = np.max(probs) + \
                              np.log(self.emission_prob(observations[t], k) + 1e-10)
        
        # Backtracking
        states = np.zeros(T, dtype=int)
        states[T-1] = np.argmax(delta[T-1])
        
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states
    
    def baum_welch(self, observations, max_iter=100, tol=1e-4):
        """Baum-Welch algorithm: learn HMM parameters."""
        T = len(observations)
        prev_log_likelihood = -np.inf
        
        for iteration in range(max_iter):
            # E-step: Forward-backward
            alpha = self.forward(observations)
            beta = self.backward(observations)
            
            # Compute gamma: P(S_t=k | O)
            gamma = alpha * beta
            gamma /= np.sum(gamma, axis=1, keepdims=True)
            
            # Compute xi: P(S_t=i, S_{t+1}=j | O)
            xi = np.zeros((T-1, self.n_states, self.n_states))
            for t in range(T-1):
                for i in range(self.n_states):
                    for j in range(self.n_states):
                        xi[t, i, j] = alpha[t, i] * \
                                      self.transition_matrix[i, j] * \
                                      self.emission_prob(observations[t+1], j) * \
                                      beta[t+1, j]
                xi[t] /= np.sum(xi[t])
            
            # M-step: Update parameters
            self.initial_probs = gamma[0]
            
            self.transition_matrix = np.sum(xi, axis=0) / \
                                     np.sum(gamma[:-1], axis=0)[:, np.newaxis]
            
            for k in range(self.n_states):
                self.emission_params['means'][k] = \
                    np.sum(gamma[:, k] * observations) / np.sum(gamma[:, k])
                self.emission_params['stds'][k] = \
                    np.sqrt(np.sum(gamma[:, k] * (observations - self.emission_params['means'][k])**2) / \
                            np.sum(gamma[:, k]))
            
            # Check convergence
            log_likelihood = np.sum(np.log(np.sum(alpha[-1])))
            
            if iteration > 0 and abs(log_likelihood - prev_log_likelihood) < tol:
                print(f'Converged at iteration {iteration}')
                break
            
            prev_log_likelihood = log_likelihood
        
        return self

print('HMM class implemented')
print('Methods: forward, backward, viterbi, baum_welch')

## 2. Generate Synthetic Regime Data

Simulate commodity returns with two regimes: low volatility and high volatility.

In [ ]:
# True HMM parameters
np.random.seed(123)
n_states = 2
T = 300

# Transition matrix (persistent regimes)
true_A = np.array([
    [0.95, 0.05],  # Low vol → Low vol, Low vol → High vol
    [0.10, 0.90]   # High vol → Low vol, High vol → High vol
])

# Emission parameters
true_means = np.array([0.001, -0.002])  # Low vol slightly positive, high vol slightly negative
true_stds = np.array([0.01, 0.03])      # Low vol low std, high vol high std

# Generate data
true_states = np.zeros(T, dtype=int)
true_states[0] = 0  # Start in low volatility

for t in range(1, T):
    true_states[t] = np.random.choice(n_states, p=true_A[true_states[t-1]])

# Generate observations
obs_returns = np.array([
    np.random.normal(true_means[true_states[t]], true_stds[true_states[t]])
    for t in range(T)
])

print(f'Generated {T} observations with {n_states} hidden states')
print(f'State 0 (low vol) frequency: {np.mean(true_states == 0):.2%}')
print(f'State 1 (high vol) frequency: {np.mean(true_states == 1):.2%}')

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(obs_returns)
axes[0].set_ylabel('Returns')
axes[0].set_title('Observed Commodity Returns')
axes[0].grid(alpha=0.3)

colors = ['green' if s == 0 else 'red' for s in true_states]
axes[1].scatter(range(T), obs_returns, c=colors, s=10, alpha=0.6)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Returns')
axes[1].set_title('True Hidden States (Green=Low Vol, Red=High Vol)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Train HMM with Baum-Welch

In [ ]:
# Initialize and train HMM
hmm = HiddenMarkovModel(n_states=2)
hmm.initialize_random()

print('Training HMM with Baum-Welch...')
hmm.baum_welch(obs_returns, max_iter=100)

print('\nLearned Parameters:')
print(f'Transition matrix:\n{hmm.transition_matrix}')
print(f'\nEmission means: {hmm.emission_params["means"]}')
print(f'Emission stds: {hmm.emission_params["stds"]}')

print('\nTrue Parameters:')
print(f'Transition matrix:\n{true_A}')
print(f'\nTrue means: {true_means}')
print(f'True stds: {true_stds}')

## 4. Decode States with Viterbi

In [ ]:
# Decode most likely state sequence
estimated_states = hmm.viterbi(obs_returns)

# Note: May need to flip labels if model learned opposite labeling
# Check which label corresponds to high volatility
if hmm.emission_params['stds'][0] > hmm.emission_params['stds'][1]:
    estimated_states = 1 - estimated_states

# Compute accuracy
accuracy = np.mean(estimated_states == true_states)
print(f'State decoding accuracy: {accuracy:.2%}')

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(obs_returns)
axes[0].set_ylabel('Returns')
axes[0].set_title('Observed Returns')
axes[0].grid(alpha=0.3)

axes[1].plot(true_states, label='True States', alpha=0.7)
axes[1].set_ylabel('State')
axes[1].set_title('True Hidden States')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(estimated_states, label='Estimated States', alpha=0.7, color='orange')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('State')
axes[2].set_title(f'Viterbi Decoded States (Accuracy={accuracy:.1%})')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Exercises

### Exercise 7.1.1: Three-State HMM

**Task:** Implement a 3-state HMM for bull/bear/sideways markets.

In [ ]:
# Exercise setup: Generate 3-state data
np.random.seed(999)
T_ex = 400

# YOUR CODE HERE
# Create and train 3-state HMM
# hmm_3state = HiddenMarkovModel(n_states=3)
# ...

hmm_3state = None  # Replace with your model

In [ ]:
def test_exercise_7_1_1():
    assert hmm_3state is not None, 'Create hmm_3state'
    assert hmm_3state.n_states == 3, 'Should have 3 states'
    assert hmm_3state.transition_matrix.shape == (3, 3), 'Check transition matrix'
    print('✅ Exercise 7.1.1 passed!')

# test_exercise_7_1_1()

### Exercise 7.1.2: Forward Algorithm

**Task:** Compute log-likelihood of observations.

In [ ]:
# YOUR CODE HERE
# Use forward algorithm to compute P(O)
# log_likelihood = ...

log_likelihood = None

In [ ]:
def test_exercise_7_1_2():
    assert log_likelihood is not None, 'Compute log_likelihood'
    assert isinstance(log_likelihood, (int, float)), 'Should be numeric'
    print(f'✅ Exercise 7.1.2 passed! Log-likelihood: {log_likelihood:.2f}')

# test_exercise_7_1_2()

---

## Summary

### Key Takeaways

1. **HMMs model hidden states**: Unobserved regimes generate observations
2. **Three algorithms**: Forward (evaluate), Viterbi (decode), Baum-Welch (learn)
3. **Regime persistence**: Transition matrices capture regime stickiness
4. **Label switching**: Learned states may have arbitrary labels
5. **Commodity applications**: Detect volatility regimes, market phases

### What's Next

- Apply HMMs to real commodity data
- Bayesian HMMs with PyMC
- Change point detection

### Additional Resources

- Rabiner (1989): "A Tutorial on Hidden Markov Models"
- Murphy (2012): "Machine Learning: A Probabilistic Perspective", Chapter 17